# P2b — Clasificación Multiclase con Softmax

**Autor:** Carlos Tessier  
**Fecha:** Marzo 2026  
**Licencia:** [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/) — Uso libre con atribución, sin fines comerciales.  

## Introducción

En los módulos de **Programación de IA** y **Sistemas de Aprendizaje Automático** es importante distinguir cuándo un problema tiene solo dos salidas posibles y cuándo exige elegir entre varias clases mutuamente excluyentes. Este cuaderno da ese paso: pasamos de la clasificación binaria que ya conocíamos a la **clasificación multiclase**.

Aquí no solo entrenaremos una red neuronal, sino que analizaremos qué cambia cuando usamos `softmax`, cómo deben codificarse las etiquetas, qué significa una matriz de confusión multiclase y cómo comparar la red con un baseline clásico como SVM.

## Objetivos de aprendizaje
- Diferenciar la clasificación binaria con `sigmoid` de la clasificación multiclase con `softmax`, relacionando cada caso con el tipo de salida esperada.
- Aplicar `sparse_categorical_crossentropy` en un problema real de varias clases y comprender qué formato deben tener las etiquetas.
- Analizar el rendimiento de un modelo multiclase mediante accuracy, curvas de entrenamiento y matriz de confusión.
- Identificar el efecto del desbalanceo de clases y relacionarlo con dificultades reales de entrenamiento y evaluación.
- Comparar una red neuronal densa con un modelo clásico multiclase para interpretar ventajas, límites y diferencias de comportamiento.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)
os.makedirs('img', exist_ok=True)

print(f"TensorFlow: {tf.__version__}")

---
## Teoría: Softmax vs Sigmoid

Hasta ahora habíamos trabajado con salidas binarias: una sola neurona decide entre dos posibilidades. En este cuaderno cambiamos de escenario: el modelo debe **repartir su decisión entre varias clases posibles**.

### Clasificación binaria (lo que ya conocemos)
- Salida: **1 neurona** con activación `sigmoid` → probabilidad de clase positiva
- Loss: `binary_crossentropy`

### Clasificación multiclase (este notebook)
- Salida: **K neuronas** con activación `softmax` → vector de probabilidades (suma = 1)

$$\text{softmax}(z_k) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

- Loss: `sparse_categorical_crossentropy` (cuando las etiquetas son enteros 0, 1, ..., K-1)  
  o `categorical_crossentropy` (cuando las etiquetas están en one-hot encoding)

La idea clave no es solo técnica, sino conceptual: en multiclase el modelo no responde sí o no a cada opción por separado, sino que **compite entre clases** para decidir cuál es la más probable.

### ¿Por qué softmax y no K sigmoids independientes?
Softmax garantiza que las probabilidades **sumen exactamente 1** y compiten entre sí.  
Con K sigmoids independientes, las probabilidades podrían sumar más o menos de 1, lo que no tiene interpretación probabilística válida.

En otras palabras: si el problema exige elegir **una clase entre varias**, `softmax` encaja mejor con la naturaleza de la tarea.

### Qué conviene tener claro antes de seguir

En este notebook vamos a tomar varias decisiones técnicas que tienen sentido solo si entendemos bien la tarea.

- Si las clases son mutuamente excluyentes, el modelo debe elegir una entre varias, no responder sí o no a cada una por separado.
- Por eso la última capa cambia respecto a clasificación binaria.
- Y por eso también cambian la función de pérdida y la forma de leer la predicción.

A partir de aquí, intenta ir relacionando cada bloque de código con esta idea general: **cómo representar correctamente un problema multiclase**.

---
## Carga y EDA — Glass Classification Dataset

El dataset Glass Classification (UCI) contiene 214 muestras de vidrio analizadas mediante sus propiedades químicas.  
El objetivo es identificar el **tipo de vidrio** a partir de 9 características físico-químicas.

| Clase | Tipo de vidrio |
|-------|----------------|
| 1 | building_windows_float_processed |
| 2 | building_windows_non_float_processed |
| 3 | vehicle_windows_float_processed |
| 5 | containers |
| 6 | tableware |
| 7 | headlamps |

*(La clase 4 no existe en este dataset)*

In [ ]:
# Cargar dataset (sin cabecera)
columnas = ['Id', 'RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']
df = pd.read_csv('data/glass.csv', header=None, names=columnas, na_values=['?'])
df = df.drop(columns=['Id']).dropna()  # el índice no es una feature

print(f"Shape: {df.shape}")
print(f"Clases presentes: {sorted(df['Type'].unique())}")
print()
df.head()

In [ ]:
# Distribución de clases
nombres_clases = {
    1: 'WF (proc)', 2: 'WNF (proc)', 3: 'WF (veh)',
    5: 'Containers', 6: 'Tableware', 7: 'Headlamps'
}

conteos = df['Type'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras de frecuencia
colores = plt.cm.Set2(np.linspace(0, 1, len(conteos)))
bars = axes[0].bar(
    [nombres_clases[c] for c in conteos.index],
    conteos.values,
    color=colores
)
axes[0].set_title('Distribución de clases')
axes[0].set_ylabel('Número de muestras')
axes[0].tick_params(axis='x', rotation=30)
for bar, v in zip(bars, conteos.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(v), ha='center', fontsize=10)

# Matriz de correlación de features
corr = df.drop(columns=['Type']).corr()
sns.heatmap(corr, ax=axes[1], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 8})
axes[1].set_title('Correlación entre features')

plt.tight_layout()
plt.savefig('img/p2b_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClase 1 y 2 dominan (vidrio de ventana). Clases 5 y 6 son muy minoritarias.")

### Qué deberías haber observado en el EDA

Antes de entrenar nada, el dataset ya nos da pistas importantes.

- No todas las clases tienen el mismo número de ejemplos.
- Eso significa que una métrica global como el accuracy puede ocultar problemas en clases minoritarias.
- También sugiere que conviene mirar la matriz de confusión al final, no solo el porcentaje de aciertos.

Esta lectura previa del dataset es parte del trabajo real en IA: no se trata solo de entrenar modelos, sino de entender si el problema está bien planteado y qué dificultades puede tener.

---
## Preprocesado

### Antes de preprocesar

Aquí vamos a hacer dos cosas muy habituales en aprendizaje automático:
- convertir las clases a enteros consecutivos, porque la función de pérdida lo necesita;
- escalar las variables, porque la red aprende mejor cuando las entradas están en rangos comparables.

Fíjate en que este paso no cambia el problema, pero sí cambia mucho la facilidad con la que el modelo puede aprender.

In [ ]:
X = df.drop(columns=['Type']).values
y = df['Type'].values

# LabelEncoder: convierte clases {1,2,3,5,6,7} → {0,1,2,3,4,5}
# Necesario porque sparse_categorical_crossentropy espera enteros 0..K-1
le = LabelEncoder()
y_enc = le.fit_transform(y)

print("Clases originales:", le.classes_)
print("Clases codificadas:", np.unique(y_enc))
print()

# Train / test split (stratified para mantener proporción de clases minoritarias)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Normalización
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

n_clases = len(le.classes_)
print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}")
print(f"Número de clases: {n_clases}")

### Qué aporta este preprocesado

Este bloque prepara el terreno para que la red pueda entrenarse de forma estable.

- `LabelEncoder` no añade información nueva: solo adapta la codificación de las clases al formato esperado por Keras.
- `StandardScaler` sí afecta al entrenamiento: evita que variables con escalas muy diferentes dominen el proceso de ajuste.
- Separar train y test antes de escalar evita fugas de información entre conjuntos.

Si este paso se hace mal, el modelo puede dar resultados engañosos aunque el código funcione.

---
## Red Neuronal Densa con Softmax

### Qué esperamos del modelo

La arquitectura sigue siendo una red densa, como en cuadernos anteriores, pero ahora la salida tiene **6 neuronas** porque hay 6 clases posibles.

La última capa con `softmax` no devuelve una única probabilidad, sino un vector completo de probabilidades. La predicción final será la clase con mayor valor.

Pregunta guía: cuando el modelo se equivoca, ¿crees que suele confundir clases totalmente distintas o clases químicamente parecidas?

In [ ]:
modelo = keras.Sequential([
    layers.Input(shape=(9,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(n_clases, activation='softmax')   # K neuronas de salida
], name='glass_multiclase')

modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',   # etiquetas como enteros
    metrics=['accuracy']
)

modelo.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=30, restore_best_weights=True
)

historial = modelo.fit(
    X_train_s, y_train,
    epochs=300,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

_, acc_nn = modelo.evaluate(X_test_s, y_test, verbose=0)
print(f"Red neuronal — Test accuracy: {acc_nn:.4f}")

### Cómo leer el entrenamiento

Después de entrenar, conviene mirar algo más que el número final de accuracy.

- Si `loss` de entrenamiento baja pero la de validación empeora, puede haber sobreajuste.
- Si ambas bajan de forma parecida, el aprendizaje suele ser más sano.
- En datasets pequeños, variaciones moderadas entre train y validation son normales.

La idea no es decidir si el modelo es perfecto, sino aprender a interpretar su comportamiento.

In [ ]:
# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, len(historial.history['loss']) + 1)

axes[0].plot(epochs, historial.history['loss'],     'b-',  label='Train', linewidth=2)
axes[0].plot(epochs, historial.history['val_loss'], 'r--', label='Val',   linewidth=2)
axes[0].set_title('Loss — sparse_categorical_crossentropy')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, historial.history['accuracy'],     'b-',  label='Train', linewidth=2)
axes[1].plot(epochs, historial.history['val_accuracy'], 'r--', label='Val',   linewidth=2)
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Red Neuronal Multiclase — Glass Dataset', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('img/p2b_training.png', dpi=150, bbox_inches='tight')
plt.show()

### Qué nos dicen las curvas

Estas gráficas ayudan a responder una pregunta clave: **¿está aprendiendo el modelo de forma útil o solo memorizando?**

No busques solo una curva bonita. Intenta relacionar lo que ves aquí con los resultados por clase que aparecerán después en la matriz de confusión.

In [ ]:
# Matriz de confusión multiclase
y_pred = np.argmax(modelo.predict(X_test_s, verbose=0), axis=1)
nombres_display = [nombres_clases[c] for c in le.classes_]

cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap de la matriz de confusión
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=nombres_display, yticklabels=nombres_display,
    linewidths=0.5
)
axes[0].set_title('Matriz de Confusión Multiclase')
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')
axes[0].tick_params(axis='x', rotation=35)
axes[0].tick_params(axis='y', rotation=0)

# Accuracy por clase
acc_por_clase = cm.diagonal() / cm.sum(axis=1)
bar_colors = ['#2ecc71' if a >= 0.8 else '#f39c12' if a >= 0.5 else '#e74c3c'
              for a in acc_por_clase]
axes[1].bar(nombres_display, acc_por_clase, color=bar_colors)
axes[1].set_title('Accuracy por Clase')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0, 1.1])
axes[1].tick_params(axis='x', rotation=35)
axes[1].axhline(y=acc_por_clase.mean(), color='blue', linestyle='--',
                label=f'Media = {acc_por_clase.mean():.2f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(acc_por_clase):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('img/p2b_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

### Cómo interpretar la matriz de confusión

Aquí aparece una de las lecturas más útiles del notebook.

- La diagonal muestra los aciertos por clase.
- Las celdas fuera de la diagonal muestran qué clases se confunden entre sí.
- Si el dataset está desbalanceado, esta matriz suele ser más informativa que el accuracy global.

La pregunta importante no es solo cuántos errores hay, sino **qué errores son**.

---
## Comparación con SVM Multiclase (Baseline)

### Por qué comparar con un baseline clásico

En IA aplicada no siempre gana el modelo más moderno. Comparar con SVM sirve para comprobar si la red neuronal realmente aporta una mejora o si un método clásico ya resuelve bien el problema.

Esta comparación también ayuda a desarrollar criterio: elegir un modelo no es solo entrenarlo, sino justificar por qué lo usas.

In [ ]:
# SVM con estrategia OneVsRest
svm = OneVsRestClassifier(SVC(kernel='rbf', C=10, gamma='scale', random_state=42))
svm.fit(X_train_s, y_train)
y_pred_svm = svm.predict(X_test_s)
acc_svm = accuracy_score(y_test, y_pred_svm)

print(f"\n{'Modelo':<25} {'Test Accuracy':>15}")
print("-" * 42)
print(f"{'SVM (OneVsRest + RBF)':<25} {acc_svm:>15.4f}")
print(f"{'Red Neuronal (softmax)':<25} {acc_nn:>15.4f}")
print()
ganador = 'Red Neuronal' if acc_nn > acc_svm else 'SVM'
print(f"Mejor modelo: {ganador}")

### Qué conclusión sacar de la comparación

Si SVM rinde igual o mejor, no significa que la red neuronal esté mal planteada. Puede significar simplemente que, para este dataset pequeño y tabular, un modelo clásico resulta muy competitivo.

Esto es una lección importante del módulo: **la elección del modelo depende del problema y de los datos**, no de modas o de complejidad aparente.

---
## Ejercicio

En esta parte interesa menos obtener un número final y más comparar comportamientos. Intenta fijarte en qué decisiones ayudan de verdad a las clases minoritarias y cuáles solo mejoran el accuracy global.

1. **Arquitectura:** El modelo actual usa `Dense(64) → Dense(64) → Dense(32) → Dense(6)`. Prueba dos variantes:
   - Modelo más profundo: añade una capa `Dense(128, relu)` al inicio
   - Modelo con Dropout: añade `layers.Dropout(0.3)` después de cada capa Dense
   ¿Cuál mejora el accuracy en clases minoritarias (Tableware, Headlamps)?

2. **Clases minoritarias:** Las clases 5 (Containers) y 6 (Tableware) tienen muy pocas muestras. Investiga qué pasa si usas `class_weight` en `model.fit()` para compensar el desbalanceo:
   ```python
   from sklearn.utils.class_weight import compute_class_weight
   pesos = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
   class_weight_dict = dict(enumerate(pesos))
   ```

3. **Reflexión:** ¿Por qué softmax en lugar de 6 sigmoids independientes para 6 clases? Explica la diferencia matemática.

Sugerencia de trabajo: en cada prueba anota tres cosas.
- qué cambio has introducido,
- qué efecto produce en validación o test,
- y si mejora realmente las clases difíciles o solo el resultado medio.

In [ ]:
# Tu código aquí


## Reflexión

Estas preguntas buscan que conectes la práctica con decisiones reales de modelado. Intenta responderlas a partir de lo que observas en las gráficas, la matriz de confusión y la comparación con SVM.

> **Pregunta 1:** La red tiene dificultades con las clases minoritarias. ¿Qué técnicas (además de `class_weight`) podrían ayudar?
>
> **Pregunta 2:** `sparse_categorical_crossentropy` vs `categorical_crossentropy`: ¿cuándo usarías cada una? ¿Qué transformación habría que aplicar a las etiquetas para pasar de una a otra?
>
> **Pregunta 3:** En la matriz de confusión, ¿qué pares de clases se confunden más? ¿Tiene sentido físicamente (propiedades químicas similares)?

*Escribe tus respuestas aquí.*

## Conclusión

En este cuaderno hemos dado el paso desde la clasificación binaria a la **clasificación multiclase**, viendo cómo cambia la salida del modelo, la función de pérdida y la forma de interpretar los resultados.

**Lo que hemos aprendido:**

- `softmax` permite modelar problemas donde las clases compiten entre sí y la suma de probabilidades debe ser 1.
- `sparse_categorical_crossentropy` resulta adecuada cuando las etiquetas están codificadas como enteros.
- La matriz de confusión multiclase es una herramienta esencial para detectar qué clases funcionan bien y cuáles presentan más errores.
- Un buen accuracy global no siempre significa buen rendimiento en todas las clases, especialmente cuando el dataset está desbalanceado.
- Comparar una red neuronal con un baseline clásico ayuda a interpretar mejor si la complejidad del modelo está justificada.

**Relación con cuadernos anteriores:**

En **P0**, **P1** y **P2** vimos cómo aprende una red, qué papel juega el gradiente descendente y cómo se entrena una red densa en tareas más simples. Aquí ampliamos ese marco a un problema con varias clases y una salida `softmax`.

**Relación con cuadernos posteriores:**

Más adelante, cuando trabajemos con CNN, transferencia, aumento de datos o NLP, seguirá siendo fundamental interpretar salidas multiclase, pérdidas categóricas, desbalanceo y matrices de confusión. Lo que cambia será el tipo de datos; la lógica de evaluación seguirá siendo muy parecida.

> **Siguiente paso:** En los próximos cuadernos aplicaremos estas ideas a arquitecturas más potentes y a datasets donde la dificultad ya no estará solo en elegir la clase correcta, sino también en extraer representaciones más ricas de los datos.